# Lab 9: Data Cleaning Pipeline

In this notebook, we apply the automated data cleaning steps onto our unstructured data analytics export. This pipeline demonstrates how missing data handling, text standardization, duplicate removal, type conversion, and automated validation logic are applied using our custom Python pipeline modules.

In [ ]:
import pandas as pd
import sys
import os
from pathlib import Path

# Ensure project root is in path
sys.path.append(os.path.abspath("../"))
from utils.logger import logger
from src.cleaning.missing_handler import report_missing

raw_csv_path = Path("../data/processed/analytics/integrated_raw_export.csv")
df_raw = pd.read_csv(raw_csv_path)
print(f"Loaded Raw Data. Shape: {df_raw.shape}")

### 1. Missing Values Handling
We report the distribution of missing values and document the steps. The missing properties are written into their own CSV.

In [ ]:
missing_report = report_missing(df_raw)
display(missing_report.head(10))

# Save missing report map
out_dir = Path("../data/processed/cleaned")
out_dir.mkdir(parents=True, exist_ok=True)
missing_report.to_csv(out_dir / "missing_report.csv")
print("Saved missing_report.csv")

### 2. String Cleaning
We use our custom robust pandas accessors within `string_cleaner` module to enforce correct casing boundaries across titles and descriptions.

In [ ]:
from src.cleaning.string_cleaner import clean_titles, normalize_language_codes, clean_overview

df_cleaned_strings = df_raw.copy()

title_cand = next((c for c in ["title", "metadata.file_name", "name"] if c in df_cleaned_strings.columns), None)
if title_cand:
    df_cleaned_strings = clean_titles(df_cleaned_strings, title_cand)

lang_cand = next((c for c in ["language", "original_language"] if c in df_cleaned_strings.columns), None)
if lang_cand:
    df_cleaned_strings = normalize_language_codes(df_cleaned_strings, lang_cand)

overview_cand = next((c for c in ["overview", "description", "summary"] if c in df_cleaned_strings.columns), None)
if overview_cand:
    df_cleaned_strings = clean_overview(df_cleaned_strings, overview_cand)

print("Sample of cleaned strings:")
cols_to_disp = [c for c in [title_cand, lang_cand, overview_cand] if c]
display(df_cleaned_strings[cols_to_disp].head(5))

### 3. Deduplication
Eliminates overlap in metadata boundaries tracking the reduction ratios per target ID properties.

In [ ]:
from src.cleaning.deduplicator import remove_exact_duplicates, remove_duplicate_ids, count_duplicates

df_dedup = df_cleaned_strings.copy()
print(f"Initial shape: {df_dedup.shape}")

exact_cnt = count_duplicates(df_dedup)
df_dedup = remove_exact_duplicates(df_dedup)
print(f"Exact duplicates removed: {exact_cnt}. Shape -> {df_dedup.shape}")

id_cand = next((c for c in ["id", "tmdb_id", "movie_id", "metadata.id"] if c in df_dedup.columns), None)
if id_cand:
    id_cnt = count_duplicates(df_dedup, [id_cand])
    df_dedup = remove_duplicate_ids(df_dedup, id_cand)
    print(f"Duplicate IDs removed: {id_cnt}. Shape -> {df_dedup.shape}")

### 4. Type Conversions
Reduces global footprint via downcasting datatypes, identifying safe memory reductions directly observable within `.info()` descriptors.

In [ ]:
from src.cleaning.type_converter import convert_to_datetime, convert_to_numeric, convert_to_category, memory_report

df_typed = df_dedup.copy()
date_cols = [c for c in ["release_date", "date", "metadata.release_date"] if c in df_typed.columns]
numeric_cols = list(df_typed.select_dtypes(include=['number']).columns)

df_typed = convert_to_datetime(df_typed, date_cols)
df_typed = convert_to_numeric(df_typed, numeric_cols)

cat_cols = [c for c in [lang_cand, "status", "source_collection"] if c and c in df_typed.columns]
df_typed = convert_to_category(df_typed, cat_cols)

mem_rep = memory_report(df_dedup, df_typed)
print(f"Memory BEFORE conversion : {mem_rep['before_mb']:.2f} MB")
print(f"Memory AFTER conversion  : {mem_rep['after_mb']:.2f} MB")
print(f"Reduction Percentage     : {mem_rep['reduction_pct']:.2f} %\n")
print(df_typed.dtypes.value_counts())

### 5. Validate & Exec Full Pipeline
Instead of staging individual segments manually, the codebase enables dynamic triggering of standard behaviors via `clean_pipeline`.

In [ ]:
from src.cleaning.clean_pipeline import run_cleaning_pipeline

df_final_clean = run_cleaning_pipeline(raw_csv_path, out_dir)
print(f"\nFull pipeline completed successfully! Cleaned DF Shape: {df_final_clean.shape}")
display(df_final_clean.head())